# check_paralle.ipynb

Run this while the coordinated sweep is running to inspect global progress without
stopping any server. It reads only the shared `out_dir`; it does not touch workers,
claims, checkpoints, or leases.

Main progress uses one deliberately simple rule:

- **done** = trainer manifest `status=done` OR `done.json` exists
- **remaining** = full sweep grid - done

Other buckets are only a breakdown of the remaining work: live in-flight claims,
expired claims that should be reclaimable, resumable partial checkpoints, failed
markers, and fresh/free combos. `vicreg_review_h5_latest.pt` alone is not counted
as done because it is also the normal resumable checkpoint written every epoch.


In [ ]:
# Config -- point at the SHARED coordinated out_dir (the same dir the current run's
# checkpoints already live in). Must match training.ipynb's OUT_DIR.
import sys, os, json, time
from pathlib import Path

REPO = '/workspace/stable-query-latent'
OUT_DIR = os.path.join(REPO, 'VICReg_review/heads/cloud_full_sweep_a100')
SWEEP_YAML = os.path.join(REPO, 'VICReg_review/sweep/sweep.yaml')
if REPO not in sys.path:
    sys.path.insert(0, REPO)

from VICReg_review.sweep.config import SweepConfig
from VICReg_review.sweep import jobspec
from VICReg_review.sweep import coordination as coord

ROOT = Path(OUT_DIR)
print('repo    :', REPO)
print('out_dir :', OUT_DIR, '(exists:', ROOT.exists(), ')')

In [ ]:
# 1. O_EXCL atomicity self-test on THIS filesystem (the claim primitive).
#    Create a file exclusively twice -> the second MUST fail. If it doesn't, the
#    network FS is not honouring exclusive create and coordination is unsafe here.
probe = ROOT / '_coord_probe'
probe.mkdir(parents=True, exist_ok=True)
tf = probe / f'oexcl_{int(time.time()*1000)}.tmp'
first = coord._atomic_create(tf, {'ok': 1})
second = coord._atomic_create(tf, {'ok': 2})
try:
    tf.unlink()
except OSError:
    pass
print('O_EXCL create: first =', first, ' second =', second)
print('VERDICT:', 'OK (exclusive create is atomic here)' if (first and not second)
      else 'FAIL -- exclusive create is NOT reliable on this mount; coordination unsafe')

In [ ]:
# 2. Coordination active? List registered VMs + whether their lease is fresh.
vm_dir = ROOT / coord.VM_DIR
vms = sorted(vm_dir.glob('*.json')) if vm_dir.exists() else []
if not vms:
    print('NO VM_parallel/ entries found ->',
          'coordination is NOT active. Either training.py is not the coordinated build,',
          'or no VM has started yet.')
else:
    now = time.time()
    print(f'{len(vms)} VM(s) registered in {vm_dir}:')
    for f in vms:
        rec = coord._read_json(f) or {}
        exp = rec.get('expiry')
        if exp is not None:
            fresh = f'lease {exp - now:+.0f}s' + (' (FRESH)' if exp > now else ' (EXPIRED)')
        else:
            age = now - f.stat().st_mtime
            fresh = f'mtime {age:.0f}s ago'
        print(f"  - {rec.get('vm', f.stem):20} pid={rec.get('pid')} {fresh}  info={rec.get('info', {})}")

In [ ]:
# 3 + 4. Global progress + remaining work.
#
# Completion is intentionally strict and simple:
#   done      = manifest status=done OR done.json
#   remaining = grid_total - done
#
# Everything else below is just a breakdown of the not-done set so you can see
# whether the remaining combos are currently training, waiting for lease reclaim,
# resumable from a partial checkpoint, failed/terminal, or not started.
from collections import Counter, defaultdict

# Heuristic "monster" buckets. These are intentionally transparent and easy to
# tweak from the notebook without touching the trainer. They mark jobs that are
# likely to be expensive on L40 or long-running even when they fit.
MONSTER_LATENTS = 1024
MONSTER_VIEW_HARD = 0.60
EDGE_LATENTS = 512
EDGE_VIEW = 0.80
LONG_TRAIN_GAMES = 1500


def read_json(path):
    try:
        return json.loads(Path(path).read_text(encoding='utf-8'))
    except Exception:
        return None


def manifest_status(cdir):
    payload = read_json(cdir / 'vicreg_review_h5_manifest.json')
    return payload.get('status') if isinstance(payload, dict) else None


def vm_is_fresh(vm_name, now=None):
    if not vm_name:
        return False
    now = time.time() if now is None else now
    rec = read_json(ROOT / coord.VM_DIR / f'{vm_name}.json')
    if not isinstance(rec, dict):
        return False
    exp = rec.get('expiry')
    if exp is not None:
        try:
            return float(exp) > now
        except Exception:
            return False
    # Legacy heartbeat-style VM records: fall back to file mtime freshness.
    try:
        return (now - (ROOT / coord.VM_DIR / f'{vm_name}.json').stat().st_mtime) <= 600
    except OSError:
        return False


def monster_tags(combo):
    tags = []
    if combo.num_latents >= MONSTER_LATENTS and combo.view >= MONSTER_VIEW_HARD:
        tags.append('A100-ish: lat1024 view>=60')
    elif combo.num_latents >= MONSTER_LATENTS and combo.view >= 0.40:
        tags.append('heavy: lat1024 view40')
    elif combo.num_latents >= EDGE_LATENTS and combo.view >= EDGE_VIEW:
        tags.append('edge: lat512 view80')

    if combo.train_games <= 0:
        tags.append('long: all games')
    elif combo.train_games >= LONG_TRAIN_GAMES:
        tags.append(f'long: n>={LONG_TRAIN_GAMES}')
    return tags


def add_sample(bucket, value, limit=12):
    if len(bucket) < limit:
        bucket.append(value)


cfg = SweepConfig.load(SWEEP_YAML)
cfg.out_dir = OUT_DIR
combos = list(cfg.iter_combos())
now = time.time()

state_counts = Counter()
live_by_vm = Counter()
expired_by_vm = Counter()
samples = defaultdict(list)
done_with_open_status = []
done_with_stale_status = []

monster_total = 0
monster_state = Counter()
monster_live_by_vm = Counter()
monster_expired_by_vm = Counter()
monster_samples = defaultdict(list)
monster_tag_total = Counter()
monster_tag_state = Counter()


def record_state(combo, state, owner=None):
    global monster_total
    state_counts[state] += 1
    add_sample(samples[state], combo.combo_id)

    tags = monster_tags(combo)
    if not tags:
        return
    monster_state[state] += 1
    add_sample(monster_samples[state], combo.combo_id)
    if state == 'live_inflight' and owner:
        monster_live_by_vm[owner] += 1
    if state == 'expired_claim' and owner:
        monster_expired_by_vm[owner] += 1
    for tag in tags:
        monster_tag_total[tag] += 1
        monster_tag_state[(tag, state)] += 1


for c in combos:
    cdir = ROOT / c.combo_id
    tags = monster_tags(c)
    if tags:
        monster_total += 1

    mstatus = manifest_status(cdir)
    has_done_json = (cdir / coord.DONE).exists()
    has_status = (cdir / coord.STATUS).exists()
    has_ckpt = (cdir / 'vicreg_review_h5_latest.pt').exists()
    has_failed = (cdir / coord.FAILED).exists()
    is_done = (mstatus == 'done') or has_done_json

    if is_done:
        record_state(c, 'done')
        if has_status:
            if has_done_json:
                done_with_open_status.append(c.combo_id)  # normal: coordinator keeps the claim marker after done.json
            else:
                done_with_stale_status.append(c.combo_id)  # trainer finished, supervisor likely died before done.json
        continue

    if has_failed:
        record_state(c, 'failed')
        continue

    if has_status:
        st = read_json(cdir / coord.STATUS) or {}
        owner = st.get('vm', '?')
        if vm_is_fresh(owner, now):
            live_by_vm[owner] += 1
            record_state(c, 'live_inflight', owner=owner)
        else:
            expired_by_vm[owner] += 1
            record_state(c, 'expired_claim', owner=owner)
        continue

    if has_ckpt:
        record_state(c, 'resuming')
        continue

    if mstatus in {'running', 'interrupted', 'error'}:
        record_state(c, 'unknown')
        continue

    record_state(c, 'free')


total = len(combos)
done = state_counts['done']
remaining = total - done
breakdown_states = ('live_inflight', 'expired_claim', 'resuming', 'failed', 'free', 'unknown')
breakdown_total = sum(state_counts[s] for s in breakdown_states)

print(f'grid      : {total} combos')
print(f'done      : {done}')
print(f'remaining : {remaining}   (= grid - done)')
print()
print('remaining breakdown:')
print(f"  live in-flight : {state_counts['live_inflight']}  by {dict(live_by_vm)}")
print(f"  expired claims : {state_counts['expired_claim']}  by {dict(expired_by_vm)}   <- should be reclaimable after lease expiry")
print(f"  resuming       : {state_counts['resuming']}   partial checkpoint, no live claim")
print(f"  failed         : {state_counts['failed']}   terminal/not done; inspect or clear separately")
print(f"  free           : {state_counts['free']}   not started yet")
print(f"  unknown        : {state_counts['unknown']}   manifest says running/interrupted/error but no status/ckpt")

if breakdown_total != remaining:
    print(f'WARNING: breakdown sums to {breakdown_total}, but remaining is {remaining}; inspect out_dir manually.')

print()
monster_done = monster_state['done']
monster_remaining = monster_total - monster_done
print('monster/heavy candidates (heuristic):')
print(f'  total     : {monster_total}')
print(f'  done      : {monster_done}')
print(f'  remaining : {monster_remaining}')
print(f"  live      : {monster_state['live_inflight']}  by {dict(monster_live_by_vm)}")
print(f"  expired   : {monster_state['expired_claim']}  by {dict(monster_expired_by_vm)}")
print(f"  resuming  : {monster_state['resuming']}")
print(f"  failed    : {monster_state['failed']}")
print(f"  free      : {monster_state['free']}")
print(f"  unknown   : {monster_state['unknown']}")
print('  by rule:')
for tag in sorted(monster_tag_total):
    t = monster_tag_total[tag]
    d = monster_tag_state[(tag, 'done')]
    live = monster_tag_state[(tag, 'live_inflight')]
    failed = monster_tag_state[(tag, 'failed')]
    rem = t - d
    print(f'    {tag:26} total={t:3d} done={d:3d} remaining={rem:3d} live={live:2d} failed={failed:2d}')

print()
if done_with_stale_status:
    print('STALE-DONE-STATUS WARNING: manifest-done combos missing done.json but still have status.json:')
    print('  ', done_with_stale_status[:20], '...' if len(done_with_stale_status) > 20 else '')
    print('  Usually means trainer wrote manifest done, then supervisor/server stopped before writing done.json.')
else:
    print('stale done/status check: OK (no manifest-only done combo has status.json)')
if done_with_open_status:
    print(f'done+status normal: {len(done_with_open_status)} combos have done.json and retained status.json')


def preview(label, values, n=10):
    if values:
        print(f'{label} sample ({min(len(values), n)}/{len(values)}):', values[:n])


print()
preview('expired claims', samples['expired_claim'])
preview('resuming', samples['resuming'])
preview('failed', samples['failed'])
preview('free', samples['free'])
preview('unknown', samples['unknown'])
preview('monster live', monster_samples['live_inflight'])
preview('monster failed', monster_samples['failed'])
preview('monster free', monster_samples['free'])
